# Common thesis evaluation

Canonical frontend for one **Architecture × Objective × Board size** cell. It runs `unified_eval_v4` and writes the same JSON, Markdown, checkpoints, and figures as the validated source evaluator.

AR uses its native trained head. JEPA trains and evaluates frozen Linear/MLP readouts, including grouped-CV head tuning. Board size defaults to **12×12**.

In [ ]:
ARCHITECTURE = "Transformer"   # "Transformer" or "Mamba"
OBJECTIVE = "AR"               # "AR" or "JEPA"
BOARD_SIZE = 12                # 8, 12, or 16
HEAD_TUNING = True
HEAD_TUNING_GAMES = 5_000
ALLOW_SOURCE_DRIFT = True      # reuse completed stages after this package refactor
OVERWRITE_INCOMPATIBLE = False
DRY_RUN = False

architecture = ARCHITECTURE.lower()
objective = OBJECTIVE.lower()
assert architecture in {"transformer", "mamba"}
assert objective in {"ar", "jepa"}
assert BOARD_SIZE in {8, 12, 16}

## Runtime

Evaluation is resumable and uses the canonical artifacts already stored in Drive. Mamba is installed only for a selected Mamba case.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from importlib import metadata as importlib_metadata
from pathlib import Path
import codecs
import os
import subprocess
import sys

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final"
)
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Master_Thesis_Artifacts")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), PROJECT_ROOT
assert ARTIFACTS_ROOT.is_dir(), ARTIFACTS_ROOT
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)
os.chdir(PROJECT_ROOT)

if architecture == "mamba":
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "packaging", "ninja",
            "causal-conv1d==1.6.2.post1",
            "mamba-ssm==2.3.2.post1",
            "--no-build-isolation",
        ],
        check=True,
    )

import torch
assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## Run the resumable evaluator

For 8×8, causal intervention is enabled automatically; JEPA 8×8 also enables the architecture-matched random-encoder control.

In [ ]:
def run_streaming(command):
    """Forward bytes immediately, including carriage-return progress updates."""
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    print("Running:", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)),
        cwd=PROJECT_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=-1,
    )
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        data = process.stdout.read1(8192)
        if not data:
            break
        sys.stdout.write(decoder.decode(data))
        sys.stdout.flush()
    tail = decoder.decode(b"", True)
    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

command = [
    sys.executable, "-u", "scripts/run_thesis_evaluation.py",
    "--architecture", architecture,
    "--objective", objective,
    "--board-size", str(BOARD_SIZE),
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--project-root", str(PROJECT_ROOT),
    "--head-tuning-games", str(HEAD_TUNING_GAMES),
]
if not HEAD_TUNING:
    command.append("--skip-head-tuning")
if ALLOW_SOURCE_DRIFT:
    command.append("--allow-source-drift")
if OVERWRITE_INCOMPATIBLE:
    command.append("--overwrite-incompatible")
if DRY_RUN:
    command.append("--dry-run")
if BOARD_SIZE == 8:
    command.append("--include-causal-intervention")
if BOARD_SIZE == 8 and objective == "jepa":
    command.append("--include-random-control")
run_streaming(command)

## Display the generated Markdown and figures

In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import Markdown, display
from othello_thesis.evaluation.thesis import resolve_case, resolve_run_dir

case = resolve_case(architecture, objective, BOARD_SIZE)
run_dir = resolve_run_dir(case, ARTIFACTS_ROOT)
report_dir = run_dir / "thesis_eval" / "final"
report_path = report_dir / "proposal_report__thesis_eval_suite_v1.md"
if DRY_RUN:
    print("Dry run complete; no report was generated.")
else:
    assert report_path.is_file(), report_path
    display(Markdown(report_path.read_text(encoding="utf-8")))
    print("Markdown:", report_path)
    for name in (
        "legal_accuracy_by_move__thesis_eval_suite_v1.png",
        "board_accuracy_by_move__thesis_eval_suite_v1.png",
        "head_saturation__thesis_eval_suite_v1.png",
    ):
        path = report_dir / name
        if path.is_file():
            display(NotebookImage(filename=str(path)))